#### Import necessary libraries

In [ ]:
try:
    import numpy as np
    import pandas as pd
    from rfdetr import RFDETRNano
    import json
    import os
    import shutil 
    from sklearn.model_selection import KFold, train_test_split
    from collections import defaultdict
    import supervision as sv
    from tqdm import tqdm
    from supervision.metrics import MeanAveragePrecision
    from PIL import Image
    from zipfile import ZipFile
except:
    !pip install numpy pandas "rfdetr == 1.2.1" scikit-learn

### PRE-WORK

Load model

In [ ]:
try:
    model = RFDETRNano()
except:
    model = RFDETRNano()

Helper functions

In [ ]:
def filter_coco_json(original_coco_data: dict, image_ids_to_keep: np.ndarray, output_path: str):
    """Creates new COCO-JSON File that only contains given Image-ID's."""
    
    new_data = {
        'images': [],
        'annotations': [],
        'categories': original_coco_data.get('categories', []),
        'info': original_coco_data.get('info', {}),
        'licenses': original_coco_data.get('licenses', [])
    }
    
    ids_set = set(image_ids_to_keep)
    
    # 1. Filter images
    kept_image_ids = set()
    for img in original_coco_data['images']:
        if img['id'] in ids_set:
            new_data['images'].append(img)
            kept_image_ids.add(img['id'])
    
    # 2. Filter annotations
    for ann in original_coco_data['annotations']:
        if ann['image_id'] in kept_image_ids:
            new_data['annotations'].append(ann)
            
    with open(output_path, 'w') as f:
        json.dump(new_data, f)
        
    return output_path


Set path

In [ ]:
path = os.getcwd()

path

Unzip the data

In [ ]:
# Create the directory if it doesn't exist
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

with ZipFile("Glass Defect Detection.v2i.coco.zip", "r") as data_set:
    data_set.extractall(extract_dir)

Initialize workflow variables

In [ ]:
N_SPLITS = 2
TEST_SIZE = 0.2
RANDOM_SEED = 42
EPOCHS = 1
MASTER_COCO_JSON_PATH = path + r"\data\train\_annotations.coco.json"
BASE_DATA_PATH = path + "/data"
TEMP_DIR = os.path.abspath("./cv_temp_coco/") 
CLASS_NAMES = ['DEFECT'] 
# --------------------------------------------------------

# Initialisierung
os.makedirs(TEMP_DIR, exist_ok=True)
all_fold_metrics = defaultdict(list)
TRAIN_POOL_JSON_PATH = os.path.join(TEMP_DIR, 'train_pool.json')
FINAL_TEST_JSON_PATH = os.path.join(TEMP_DIR, 'test_final.json')

In [ ]:
print("1. Loading Master COCO JSON and execute 80/20 Split...")
with open(MASTER_COCO_JSON_PATH, 'r') as f:
    master_coco_data = json.load(f)

# Map Image ID to Filename for efficient mapping
image_id_to_filename = {img['id']: img['file_name'] for img in master_coco_data['images']}

X_ids = np.array(list(image_id_to_filename.keys()))
y_dummy = np.zeros(len(X_ids)) 

# 80/20 Split of IDs 
X_pool_ids, X_test_ids, _, _ = train_test_split(
    X_ids, y_dummy, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_SEED, 
    shuffle=True
)

# Create permanent JSON-Files
filter_coco_json(master_coco_data, X_pool_ids, TRAIN_POOL_JSON_PATH)
filter_coco_json(master_coco_data, X_test_ids, FINAL_TEST_JSON_PATH)

print(f"Split done: Train-Pool ({len(X_pool_ids)} Images) and Test-Final ({len(X_test_ids)} Images)")

### Training Logic

In [ ]:
# Load training Pool for fold generation
with open(TRAIN_POOL_JSON_PATH, 'r') as f:
    train_pool_data = json.load(f)

X_pool_ids = np.array([img['id'] for img in train_pool_data['images']])

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

best_mAP = 0.0
best_model_run_name = ""


for fold, (train_index, val_index) in enumerate(kf.split(X_pool_ids)):
    print(f"\n==================================================")
    print(f"--- FOLD {fold+1}/{N_SPLITS} started... ---")
    print(f"==================================================")

    X_train_ids = X_pool_ids[train_index]
    X_val_ids = X_pool_ids[val_index]

    # 1. Create temporary folder structure (train/, valid/, test/)
    train_fold_dir = os.path.join(TEMP_DIR, 'train')
    val_fold_dir = os.path.join(TEMP_DIR, 'valid') 
    test_fold_dir = os.path.join(TEMP_DIR, 'test') 
    
    # clean and creation of structure
    for d in [train_fold_dir, val_fold_dir, test_fold_dir]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

    # 2. Create JSON-File with expected name
    train_json_path = os.path.join(train_fold_dir, '_annotations.coco.json')
    val_json_path = os.path.join(val_fold_dir, '_annotations.coco.json')
    test_json_path = os.path.join(test_fold_dir, '_annotations.coco.json') # Dummy-path -> it wont work without the dummy path because of the library mechanism
    
    filter_coco_json(train_pool_data, X_train_ids, train_json_path)
    filter_coco_json(train_pool_data, X_val_ids, val_json_path)
    
    # creation of empty Dummy-JSON for 'test'-Split
    dummy_test_data = {
        'images': [], 'annotations': [], 'categories': master_coco_data['categories']
    }
    with open(test_json_path, 'w') as f:
        json.dump(dummy_test_data, f)
        
    # 3. Copy images (Otherwise it wont find the images)
    
    def copy_images_to_fold(id_list, dest_dir):
        """Copies images based on their ID in the right folder"""
        for image_id in id_list:
            file_name = image_id_to_filename[image_id]
            source_path = os.path.join(BASE_DATA_PATH + "/train", file_name)
            dest_path = os.path.join(dest_dir, file_name)
            
            try:
                # Kopiert die Datei vom Original-Speicherort in den temporären Fold-Ordner
                shutil.copyfile(source_path, dest_path)
            except FileNotFoundError:
                print(f"WARINING: Image not found {source_path}. Skipping image.")
            except Exception as e:
                print(f"ERROR while copying {file_name}: {e}")

    copy_images_to_fold(X_train_ids, train_fold_dir)
    copy_images_to_fold(X_val_ids, val_fold_dir)


    # 4. RF-DETR Training 
    run_name = f'rf_detr_cv_fold_{fold+1}'
    
    results = model.train(
        dataset_dir=TEMP_DIR, 
        epochs=EPOCHS,
        name=run_name,
        project='./rf-detr_runs',
        val=True,
        run_test=False, 
    )


# FOLD RESULTS NEED TO BE EXTRACTED MANUALLY FROM THE TRAINING HISTORY

### Testing

In [ ]:
ds = sv.DetectionDataset.from_coco(
    images_directory_path=BASE_DATA_PATH + "/train",
    annotations_path=f"{path}/cv_temp_coco/test_final.json",
)

model = RFDETRNano(pretrain_weights=f"{path}/output/checkpoint_best_total.pth")
model.optimize_for_inference()


targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)

    targets.append(annotations)
    predictions.append(detections)

map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)


# Clean (removes copied images)
shutil.rmtree(train_fold_dir)
shutil.rmtree(val_fold_dir)
shutil.rmtree(test_fold_dir)

In [ ]:
map_result.plot()